# Detectron2 — Research-Grade Object Detection & Segmentation

## What Is This Notebook About?

YOLO draws rectangles around objects. **Detectron2** goes further — it can tell you the *exact shape* of every object at the pixel level, recognize each individual *instance* of an object separately, and detect 17 body *keypoints* on every person.

Detectron2 is Facebook AI Research's production-level computer vision library. It powers Instagram's AI features, Facebook's photo tagging, and real-world research in self-driving cars and medical imaging.

---

## Detection Taxonomy (What Each Task Does)

```
Input image of 3 people standing together:

Object Detection:     [BOX around all 3 people together] → 'person' × 3
Semantic Segmentation: All person pixels = one color, background = another
Instance Segmentation: Person 1 pixels = red, Person 2 = blue, Person 3 = green
Panoptic Segmentation: Instance seg for objects + semantic seg for background
Pose Estimation:       17 keypoints (joints) per person
```

---

## Real-World Applications

| Task | Application |
|---|---|
| Instance segmentation | AR effects, background removal, photo editing |
| Pose estimation | Sports analytics, physical therapy, gaming controls |
| Panoptic segmentation | Autonomous driving scene understanding |
| Object detection | Same as YOLO — Detectron2 has higher-accuracy models |

---

## Prerequisites

- OpenCV notebook (image basics)
- Torchvision notebook (CNNs, transfer learning)
- YOLO notebook (object detection concepts, IoU, NMS, mAP)

---

## Table of Contents

1. [Setup & Installation](#1-setup)
2. [Detectron2 Architecture — Mask R-CNN Explained](#2-architecture)
3. [Detection: Semantic vs Instance vs Panoptic](#3-segmentation-types)
4. [Running Inference — DefaultPredictor](#4-inference)
5. [The Config System](#5-config)
6. [Visualizing Results with Visualizer](#6-visualizer)
7. [Model Zoo — All Pre-trained Models](#7-model-zoo)
8. [Training on Custom Data — Registering a Dataset](#8-training)
9. [Mini Project — Medical Cell Instance Segmentation](#9-mini-project)
10. [Detectron2 vs YOLO — When to Use Which](#10-comparison)
11. [Common Pitfalls](#11-pitfalls)
12. [Interview Q&A](#12-interview)
13. [Resources](#13-resources)
14. [Summary](#14-summary)

---
## 1. Setup & Installation <a id='1-setup'></a>

### Important Note on Installation

Detectron2 must be built from source (no pip wheel for all platforms). Installation depends on your PyTorch and CUDA version.

```bash
# Install PyTorch first (match your CUDA version)
pip install torch torchvision

# Install Detectron2 (pre-built wheel — fastest)
pip install 'git+https://github.com/facebookresearch/detectron2.git'

# OR: install pre-built wheel (check https://detectron2.readthedocs.io/en/latest/tutorials/install.html)
# python -m pip install detectron2 -f \
#   https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/index.html
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import warnings
warnings.filterwarnings('ignore')

# Try importing Detectron2
try:
    import detectron2
    from detectron2 import model_zoo
    from detectron2.config import get_cfg
    from detectron2.engine import DefaultPredictor, DefaultTrainer
    from detectron2.utils.visualizer import Visualizer, ColorMode
    from detectron2.data import MetadataCatalog, DatasetCatalog
    from detectron2.data.datasets import register_coco_instances
    from detectron2.structures import BoxMode
    import torch
    D2_AVAILABLE = True
    print(f"Detectron2 version: {detectron2.__version__}")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Device: {device}")
except ImportError as e:
    D2_AVAILABLE = False
    print("Detectron2 not installed.")
    print(f"Error: {e}")
    print("Install: pip install 'git+https://github.com/facebookresearch/detectron2.git'")
    print("\nAll code blocks show real API usage.")
    print("Output is simulated for learning purposes.")

np.random.seed(42)
print("\nReady!")

---
## 2. Detectron2 Architecture — Mask R-CNN Explained <a id='2-architecture'></a>

### The Assembly Line Analogy

Mask R-CNN (the flagship Detectron2 model) works like a car assembly line with specialized stations:

**Station 1 — Backbone (Feature Extractor):**
ResNet-50 or ResNet-101 scans the image and extracts feature maps at multiple scales.

**Station 2 — FPN (Feature Pyramid Network):**
Combines features from multiple scales — big objects use low-resolution features, small objects use high-resolution features. Critical for detecting objects of different sizes.

**Station 3 — RPN (Region Proposal Network):**
Scans the feature map and proposes ~300 candidate regions ("there might be an object here"). Much faster than sliding windows.

**Station 4 — RoI Align:**
For each proposed region, extracts a fixed-size feature map from the appropriate FPN level. Uses bilinear interpolation (no rounding artifacts).

**Station 5 — Box Head:**
Refines the bounding box coordinates and predicts class probabilities.

**Station 6 — Mask Head (Mask R-CNN only):**
A small FCN (Fully Convolutional Network) that predicts a 28×28 binary mask for each detected object.

```
Image
  → ResNet backbone (feature extraction)
  → FPN (multi-scale features)
  → RPN (propose candidate regions)
  → RoI Align (extract per-region features)
  → Box Head → (refined box, class)
  → Mask Head → (28×28 binary mask per instance)
  → Final output: (box, class, score, mask) × N instances
```

**Research paper:** https://arxiv.org/abs/1703.06870 (He et al., 2017)

In [ ]:
# ── Mask R-CNN architecture diagram ──────────────────────────────────────────

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16); ax.set_ylim(0, 6)
ax.axis('off')

boxes_data = [
    (0.2, 1.5, 2.0, 3.0, '#3498db', 'Input\nImage\n(H×W×3)'),
    (2.5, 1.5, 2.2, 3.0, '#e74c3c', 'ResNet\nBackbone\n(feature maps)'),
    (5.0, 1.5, 1.8, 3.0, '#e67e22', 'FPN\n(multi-scale\nfeatures)'),
    (7.1, 1.5, 1.8, 3.0, '#9b59b6', 'RPN\n(region\nproposals)'),
    (9.2, 1.5, 1.8, 3.0, '#27ae60', 'RoI\nAlign\n(crop+resize)'),
    (11.3, 2.8, 1.8, 1.5, '#2980b9', 'Box\nHead'),
    (11.3, 1.0, 1.8, 1.5, '#c0392b', 'Mask\nHead\n(28×28)'),
    (13.4, 2.0, 1.8, 2.0, '#16a085', 'Output:\nboxes +\nmasks'),
]

for x, y, w, h, color, label in boxes_data:
    rect = patches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                                   facecolor=color, alpha=0.85, edgecolor='white', lw=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
           fontsize=8, color='white', fontweight='bold')

# Arrows
arrow_props = dict(arrowstyle='->', color='#2c3e50', lw=2)
connections = [(2.2, 3.0), (4.7, 3.0), (6.8, 3.0), (8.9, 3.0), (11.1, 3.55), (11.1, 1.75), (13.1, 3.0)]
for i in range(len(connections)-1):
    x1, y1 = connections[i]
    x2, y2 = connections[i+1]
    if i < 4:
        ax.annotate('', xy=(x2-0.05, y2), xytext=(x1, y1), arrowprops=arrow_props)

# Branch from RoI Align to both heads
ax.annotate('', xy=(11.1, 3.55), xytext=(11.0, 3.0), arrowprops=arrow_props)
ax.annotate('', xy=(11.1, 1.75), xytext=(11.0, 3.0), arrowprops=arrow_props)
ax.annotate('', xy=(13.2, 3.0), xytext=(13.1, 3.55), arrowprops=arrow_props)
ax.annotate('', xy=(13.2, 2.5), xytext=(13.1, 1.75), arrowprops=arrow_props)

ax.set_title('Mask R-CNN Architecture (Detectron2 flagship model)\n'
             'Two-stage: RPN proposes regions → heads refine + segment', fontsize=13)
plt.tight_layout()
plt.show()

print("Key architectural advantages over YOLO:")
print("  1. FPN handles objects at MULTIPLE scales simultaneously")
print("  2. RoI Align gives precise per-instance feature extraction")
print("  3. Separate mask head gives pixel-precise instance segmentation")
print("  4. Two-stage design → higher accuracy (at cost of speed)")

---
## 3. Detection: Semantic vs Instance vs Panoptic <a id='3-segmentation-types'></a>

In [ ]:
# ── Visualize the difference between segmentation types ──────────────────────

def create_multi_person_scene():
    """Synthetic scene with 3 people and background for demonstrating seg types."""
    img = np.ones((300, 500, 3), dtype=np.uint8) * 180  # gray background

    # Sky
    img[:150, :] = [200, 220, 240]
    # Ground
    img[200:, :] = [130, 160, 100]

    # Person 1 (left)
    cv2.circle(img, (100, 80), 25, (200, 170, 140), -1)     # head
    cv2.rectangle(img, (80, 105), (120, 185), (60, 100, 180), -1)  # body
    cv2.rectangle(img, (73, 185), (95, 250), (40, 60, 120), -1)    # left leg
    cv2.rectangle(img, (103, 185), (125, 250), (40, 60, 120), -1)  # right leg

    # Person 2 (center)
    cv2.circle(img, (260, 75), 25, (170, 140, 120), -1)
    cv2.rectangle(img, (240, 100), (280, 185), (180, 50, 50), -1)
    cv2.rectangle(img, (233, 185), (255, 250), (120, 30, 30), -1)
    cv2.rectangle(img, (263, 185), (285, 250), (120, 30, 30), -1)

    # Person 3 (right)
    cv2.circle(img, (410, 80), 25, (210, 180, 150), -1)
    cv2.rectangle(img, (390, 105), (430, 185), (50, 160, 80), -1)
    cv2.rectangle(img, (383, 185), (405, 250), (30, 110, 50), -1)
    cv2.rectangle(img, (413, 185), (435, 250), (30, 110, 50), -1)

    return img


scene = create_multi_person_scene()

# Simulate: Object Detection
det_img = scene.copy()
person_boxes = [(70, 55, 135, 255), (230, 50, 295, 255), (380, 55, 445, 255)]
for x1, y1, x2, y2 in person_boxes:
    cv2.rectangle(det_img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    cv2.putText(det_img, 'person', (x1, y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 1)

# Simulate: Semantic Segmentation — all person pixels same color
sem_img = scene.copy()
sem_overlay = np.zeros_like(scene)
for x1, y1, x2, y2 in person_boxes:
    cv2.rectangle(sem_overlay, (x1+5, y1+5), (x2-5, y2-5), (0, 0, 200), -1)
# Background: different color
bg_mask = np.ones((300, 500), dtype=bool)
for x1, y1, x2, y2 in person_boxes:
    bg_mask[y1:y2, x1:x2] = False
sem_overlay[bg_mask] = (100, 160, 100)  # green for background
sem_img = cv2.addWeighted(scene, 0.4, sem_overlay, 0.6, 0)
cv2.putText(sem_img, 'red=person, green=background', (5, 290), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)

# Simulate: Instance Segmentation — each person different color
inst_img = scene.copy()
inst_overlay = np.zeros_like(scene)
instance_colors = [(200, 50, 50), (50, 50, 200), (50, 200, 50)]
for (x1, y1, x2, y2), color in zip(person_boxes, instance_colors):
    cv2.rectangle(inst_overlay, (x1+5, y1+5), (x2-5, y2-5), color, -1)
inst_img = cv2.addWeighted(scene, 0.5, inst_overlay, 0.5, 0)
for (x1, y1, x2, y2), color in zip(person_boxes, instance_colors):
    cv2.rectangle(inst_img, (x1, y1), (x2, y2), color, 2)
cv2.putText(inst_img, 'each person = unique color', (5, 290), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)

# Simulate: Pose Estimation
pose_img = scene.copy()
person_keypoints = [
    {'head': (100,80), 'l_sh': (80,105), 'r_sh': (120,105), 'l_hip': (83,185), 'r_hip': (117,185)},
    {'head': (260,75), 'l_sh': (240,100),'r_sh': (280,100),'l_hip': (243,185),'r_hip': (277,185)},
    {'head': (410,80), 'l_sh': (390,105),'r_sh': (430,105),'l_hip': (393,185),'r_hip': (427,185)},
]
skel = [('head','l_sh'),('head','r_sh'),('l_sh','r_sh'),('l_sh','l_hip'),('r_sh','r_hip'),('l_hip','r_hip')]
kp_color = [(0,200,255), (0,150,255), (0,100,200)]
for kps, kc in zip(person_keypoints, kp_color):
    for a, b in skel:
        if a in kps and b in kps:
            cv2.line(pose_img, kps[a], kps[b], kc, 2)
    for pt in kps.values():
        cv2.circle(pose_img, pt, 5, (255,255,0), -1)

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
images_to_show = [scene, det_img, sem_img, inst_img, pose_img]
titles = ['Original', 'Object Detection\n(bounding boxes)',
          'Semantic Segmentation\n(class-level pixels)',
          'Instance Segmentation\n(per-instance pixels)',
          'Pose Estimation\n(body keypoints)']

for ax, img, title in zip(axes, images_to_show, titles):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.suptitle('Computer Vision Tasks Hierarchy — Same Scene, Different Outputs', fontsize=13)
plt.tight_layout()
plt.show()

print("\nWhen to use which:")
print("  Object Detection:      Fast, sufficient for counting/tracking")
print("  Semantic Segmentation: Autonomous driving scene understanding")
print("  Instance Segmentation: Each object individually (AR, medical imaging)")
print("  Panoptic Segmentation: Both stuff (sky, road) + things (cars, people)")
print("  Pose Estimation:       Sports, physical therapy, AR body effects")

---
## 4. Running Inference — DefaultPredictor <a id='4-inference'></a>

In [ ]:
# ── Detectron2 inference with DefaultPredictor ────────────────────────────────

if D2_AVAILABLE:
    # ── Step 1: Create config ──────────────────────────────────────────────────
    cfg = get_cfg()

    # Merge from Model Zoo config file
    cfg.merge_from_file(
        model_zoo.get_config_file(
            'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
        )
    )

    # Set pre-trained weights
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
        'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
    )

    # Inference threshold
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
    cfg.MODEL.DEVICE = device

    # ── Step 2: Create predictor ───────────────────────────────────────────────
    predictor = DefaultPredictor(cfg)
    print("DefaultPredictor created with Mask R-CNN R50-FPN")

    # ── Step 3: Run inference ──────────────────────────────────────────────────
    # Save synthetic scene
    cv2.imwrite('/tmp/scene_d2.jpg', scene)
    im = cv2.imread('/tmp/scene_d2.jpg')  # Detectron2 expects BGR uint8

    outputs = predictor(im)

    # ── Step 4: Inspect outputs ────────────────────────────────────────────────
    instances = outputs['instances']
    print(f"\nDetected {len(instances)} instances")
    print(f"Detected fields: {instances.get_fields().keys()}")
    print(f"  pred_boxes:    {instances.pred_boxes.tensor.shape}  (N, 4)")
    print(f"  scores:        {instances.scores.shape}  (N,)")
    print(f"  pred_classes:  {instances.pred_classes.shape}  (N,)")
    if instances.has('pred_masks'):
        print(f"  pred_masks:    {instances.pred_masks.shape}  (N, H, W)")

    # Get class names
    metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])
    class_names = metadata.thing_classes

    for i in range(len(instances)):
        box   = instances.pred_boxes.tensor[i].tolist()
        score = float(instances.scores[i])
        cls   = int(instances.pred_classes[i])
        print(f"  [{score:.3f}] {class_names[cls]:15s} at ({box[0]:.0f},{box[1]:.0f},{box[2]:.0f},{box[3]:.0f})")

else:
    print("[CODE REFERENCE — install Detectron2 to run]")
    print()
    print("# Complete inference pipeline:")
    print("""
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
import cv2

# 1. Configure model
cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file(
        'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
    )
)
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5   # confidence threshold
cfg.MODEL.DEVICE = 'cuda'   # or 'cpu'

# 2. Create predictor (loads weights)
predictor = DefaultPredictor(cfg)

# 3. Run on image (BGR numpy array, uint8)
im = cv2.imread('image.jpg')
outputs = predictor(im)

# 4. Inspect results
instances = outputs['instances']   # all detected objects
instances.pred_boxes      # Boxes object → .tensor shape (N,4)
instances.scores          # tensor (N,): confidence per instance
instances.pred_classes    # tensor (N,): class index per instance
instances.pred_masks      # tensor (N,H,W): binary mask per instance

# Filter to CPU for numpy operations
instances_cpu = instances.to('cpu')
boxes = instances_cpu.pred_boxes.tensor.numpy()   # (N,4)
masks = instances_cpu.pred_masks.numpy()          # (N,H,W) bool
""")

---
## 5. The Config System <a id='5-config'></a>

### The Recipe Book Analogy

Detectron2's config system is like a recipe book: each config file is a recipe (architecture + hyperparameters). You can merge multiple recipes and override specific ingredients.

In [ ]:
# ── Config system overview ────────────────────────────────────────────────────

print("="*65)
print("DETECTRON2 CONFIG SYSTEM")
print("="*65)

print("""
from detectron2.config import get_cfg
from detectron2 import model_zoo

cfg = get_cfg()   # empty config with all defaults

# Merge a model zoo config (sets architecture, training schedule, etc.)
cfg.merge_from_file(model_zoo.get_config_file(
    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
))

# Override specific values
cfg.DATASETS.TRAIN = ('my_dataset_train',)    # registered dataset name
cfg.DATASETS.TEST  = ('my_dataset_val',)
cfg.DATALOADER.NUM_WORKERS = 4

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(...)  # pre-trained weights
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 3     # number of your custom classes
cfg.MODEL.DEVICE = 'cuda'

cfg.SOLVER.IMS_PER_BATCH = 2         # images per batch (reduce if OOM)
cfg.SOLVER.BASE_LR = 0.00025         # learning rate
cfg.SOLVER.MAX_ITER = 3000           # training iterations
cfg.SOLVER.STEPS = (2000, 2500)      # LR decay at these iterations
cfg.SOLVER.CHECKPOINT_PERIOD = 500   # save checkpoint every N iterations

cfg.INPUT.MIN_SIZE_TRAIN = (640, 672, 704, 736, 768, 800)  # multi-scale

cfg.OUTPUT_DIR = './output/my_experiment'

# Save config to file (reproducibility)
import os
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
with open(os.path.join(cfg.OUTPUT_DIR, 'config.yaml'), 'w') as f:
    f.write(cfg.dump())
""")

if D2_AVAILABLE:
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(
        'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
    ))
    print("\nKey config values:")
    print(f"  MODEL.BACKBONE.NAME:              {cfg.MODEL.BACKBONE.NAME}")
    print(f"  MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE: {cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE}")
    print(f"  SOLVER.BASE_LR:                   {cfg.SOLVER.BASE_LR}")
    print(f"  SOLVER.MAX_ITER:                  {cfg.SOLVER.MAX_ITER}")
    print(f"  INPUT.MIN_SIZE_TRAIN:             {cfg.INPUT.MIN_SIZE_TRAIN}")

---
## 6. Visualizing Results with Visualizer <a id='6-visualizer'></a>

In [ ]:
# ── Visualizer API reference ──────────────────────────────────────────────────

if D2_AVAILABLE:
    im = cv2.imread('/tmp/scene_d2.jpg')
    metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])

    # Default mode: uses instance-specific random colors
    v = Visualizer(
        im[:, :, ::-1],  # BGR → RGB for Visualizer
        metadata=metadata,
        scale=1.5,
        instance_mode=ColorMode.IMAGE_BW  # grayscale + colored instances
    )
    out = v.draw_instance_predictions(outputs['instances'].to('cpu'))
    result_img = out.get_image()  # RGB numpy array

    plt.figure(figsize=(12, 8))
    plt.imshow(result_img)
    plt.title('Detectron2 Instance Segmentation Result', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

else:
    print("[CODE REFERENCE — Visualizer API]")
    print("""
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import MetadataCatalog

im = cv2.imread('image.jpg')  # BGR
metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])

# Create visualizer
v = Visualizer(
    im[:, :, ::-1],        # BGR → RGB
    metadata=metadata,
    scale=1.0,
    instance_mode=ColorMode.IMAGE        # full color image
    # instance_mode=ColorMode.IMAGE_BW   # grayscale + colored instances
    # instance_mode=ColorMode.SEGMENTATION  # color by class
)

# Draw predictions
out = v.draw_instance_predictions(outputs['instances'].to('cpu'))
result = out.get_image()   # RGB numpy array

# Show or save
plt.imshow(result)
plt.show()

# Or draw ground truth dataset annotations
out2 = v.draw_dataset_dict(dataset_dict)
""")

    # Show simulated visualization
    seg_vis = scene.copy()
    seg_overlay = np.zeros_like(scene)
    sim_colors = [(80, 50, 200), (50, 200, 80), (200, 80, 50)]
    for (x1,y1,x2,y2), color in zip(person_boxes, sim_colors):
        cv2.rectangle(seg_overlay, (x1+5, y1+5), (x2-5, y2-5), color, -1)
        cv2.rectangle(seg_vis, (x1, y1), (x2, y2), color, 3)
        cv2.putText(seg_vis, 'person', (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    sim_result = cv2.addWeighted(seg_vis, 0.55, seg_overlay, 0.45, 0)

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(sim_result, cv2.COLOR_BGR2RGB))
    plt.title('[SIMULATED] Detectron2 Instance Segmentation\n'
              'Each instance gets a unique color mask + bounding box', fontsize=11)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

---
## 7. Model Zoo — All Pre-trained Models <a id='7-model-zoo'></a>

In [ ]:
# ── Detectron2 Model Zoo overview ─────────────────────────────────────────────

model_zoo_overview = {
    'Object Detection': [
        ('Faster R-CNN R50-FPN-1x',  'COCO-Detection/faster_rcnn_R_50_FPN_1x.yaml',         37.9, 'Fast'),
        ('Faster R-CNN R50-FPN-3x',  'COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml',         40.2, 'Good balance'),
        ('Faster R-CNN R101-FPN-3x', 'COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml',        42.0, 'Higher accuracy'),
        ('RetinaNet R50-FPN-1x',     'COCO-Detection/retinanet_R_50_FPN_1x.yaml',           36.5, 'One-stage, fast'),
    ],
    'Instance Segmentation': [
        ('Mask R-CNN R50-FPN-1x',    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_1x.yaml', 38.6, 'Fast'),
        ('Mask R-CNN R50-FPN-3x',    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml', 41.0, '← Start here'),
        ('Mask R-CNN R101-FPN-3x',   'COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml',42.9, 'High accuracy'),
        ('Mask R-CNN X101-32x8d',    'COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml',44.3,'Maximum accuracy'),
    ],
    'Panoptic Segmentation': [
        ('Panoptic FPN R50',         'COCO-PanopticSegmentation/panoptic_fpn_R_50_3x.yaml', 41.5, 'Standard'),
        ('Panoptic FPN R101',        'COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml',43.0, 'Higher accuracy'),
    ],
    'Keypoint Detection': [
        ('Keypoint R50-FPN-1x',      'COCO-Keypoints/keypoint_rcnn_R_50_FPN_1x.yaml',       64.0, 'Pose estimation'),
        ('Keypoint R101-FPN-3x',     'COCO-Keypoints/keypoint_rcnn_R_101_FPN_3x.yaml',      66.1, 'Higher accuracy'),
    ],
}

for category, models in model_zoo_overview.items():
    print(f"\n{'─'*65}")
    print(f"  {category}")
    print(f"{'─'*65}")
    print(f"  {'Model':30s} {'mAP/AP':8s} {'Notes'}")
    for name, config, ap, notes in models:
        print(f"  {name:30s} {ap:8.1f} {notes}")

print("""

Loading any model:

  from detectron2 import model_zoo
  from detectron2.config import get_cfg
  from detectron2.engine import DefaultPredictor

  cfg = get_cfg()
  # Swap config file → swap model
  cfg.merge_from_file(model_zoo.get_config_file(
      'COCO-Keypoints/keypoint_rcnn_R_50_FPN_1x.yaml'
  ))
  cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
      'COCO-Keypoints/keypoint_rcnn_R_50_FPN_1x.yaml'
  )
  cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
  predictor = DefaultPredictor(cfg)

  outputs = predictor(im)
  keypoints = outputs['instances'].pred_keypoints  # (N, 17, 3)
  # 3 values per keypoint: x, y, confidence
""")

---
## 8. Training on Custom Data — Registering a Dataset <a id='8-training'></a>

Detectron2 uses a **dataset registry** system. You write a function that returns your dataset in a standard format, register it by name, then reference that name in your config.

In [ ]:
# ── Custom dataset registration ───────────────────────────────────────────────

print("="*65)
print("TRAINING ON CUSTOM DATA — FULL PIPELINE")
print("="*65)

print("""
import os, json, random
import numpy as np
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.structures import BoxMode


# STEP 1: Write a dataset loader function
# It must return a list of dicts, one per image
def get_my_dataset_dicts(image_dir, annotation_file):
    """Load annotations and return list of dataset dicts."""
    with open(annotation_file) as f:
        annotations = json.load(f)

    dataset_dicts = []
    for ann in annotations:
        record = {
            'file_name':    os.path.join(image_dir, ann['filename']),
            'image_id':     ann['id'],
            'height':       ann['height'],
            'width':        ann['width'],
            'annotations':  []
        }

        for obj in ann['objects']:
            bbox = obj['bbox']   # [x_min, y_min, width, height]
            obj_dict = {
                'bbox':       bbox,
                'bbox_mode':  BoxMode.XYWH_ABS,   # or XYXY_ABS
                'category_id': obj['category_id'],
                # For segmentation, add polygon or RLE mask:
                'segmentation': [obj['polygon']],  # list of [x1,y1,x2,y2,...]
            }
            record['annotations'].append(obj_dict)

        dataset_dicts.append(record)

    return dataset_dicts


# STEP 2: Register the dataset
for split in ['train', 'val']:
    DatasetCatalog.register(
        f'my_dataset_{split}',
        lambda s=split: get_my_dataset_dicts(
            f'data/{s}/images',
            f'data/{s}/annotations.json'
        )
    )
    MetadataCatalog.get(f'my_dataset_{split}').set(
        thing_classes=['cell', 'nucleus', 'debris']
    )


# STEP 3: Configure and train
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2 import model_zoo

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
))

cfg.DATASETS.TRAIN = ('my_dataset_train',)
cfg.DATASETS.TEST  = ('my_dataset_val',)
cfg.DATALOADER.NUM_WORKERS = 4

# Start from COCO pre-trained weights
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
)

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128   # 256 is default, 128 for small datasets
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 3              # your class count
cfg.MODEL.DEVICE = 'cuda'

cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 1000
cfg.SOLVER.STEPS = (700, 900)

cfg.OUTPUT_DIR = './output/my_model'
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Train!
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()


# STEP 4: Evaluate
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

evaluator = COCOEvaluator('my_dataset_val', output_dir='./output/eval')
val_loader = build_detection_test_loader(cfg, 'my_dataset_val')
print(inference_on_dataset(trainer.model, val_loader, evaluator))
""")

print("\nTIP: Use COCO format (via cvat/labelstudio) for easiest Detectron2 integration:")
print("  DatasetCatalog.register only needs one line with COCO JSON:")
print("  register_coco_instances('my_dataset_train', {}, 'train.json', 'train_images/')")

---
## 9. Mini Project — Medical Cell Instance Segmentation <a id='9-mini-project'></a>

### What We're Building

A pipeline that segments individual cells in a microscopy image — each cell gets its own mask, enabling:
- Automated cell counting
- Area/shape measurement per cell
- Identifying abnormal cells by shape

**Real-world equivalent:** Pathology labs use this to analyze blood smears, tumor biopsies, and Pap smear slides.

In [ ]:
# ── Simulate synthetic microscopy data ────────────────────────────────────────

np.random.seed(99)

def create_cell_image(n_cells=20, img_size=512):
    """Generate synthetic fluorescence microscopy image with labeled cells."""
    # Dark background (microscopy)
    img = np.zeros((img_size, img_size, 3), dtype=np.uint8)

    # Add some background noise/texture
    bg_noise = np.random.poisson(5, (img_size, img_size)).astype(np.uint8)
    img[:, :, 1] = np.clip(bg_noise, 0, 255)  # slight green channel

    cells = []   # list of (center_x, center_y, radius, type)

    def overlaps(cx, cy, r, existing, margin=5):
        for ex, ey, er, _ in existing:
            if np.sqrt((cx-ex)**2 + (cy-ey)**2) < r + er + margin:
                return True
        return False

    cell_types = [
        ('normal',   0.65, (0, 200, 50), 15, 28),    # green, medium
        ('abnormal', 0.20, (0, 60, 200), 25, 40),    # blue, large irregular
        ('dead',     0.15, (100, 20, 0), 8, 15),     # dark red, small
    ]

    for _ in range(n_cells):
        # Choose type
        r_val = np.random.rand()
        cumulative = 0
        for ctype, prob, color, rmin, rmax in cell_types:
            cumulative += prob
            if r_val < cumulative:
                break

        r = np.random.randint(rmin, rmax)
        for _ in range(100):
            cx = np.random.randint(r+10, img_size-r-10)
            cy = np.random.randint(r+10, img_size-r-10)
            if not overlaps(cx, cy, r, cells):
                cells.append((cx, cy, r, ctype))

                # Draw cell body
                intensity_var = np.random.randint(-30, 30, 3)
                cell_color = tuple(np.clip(np.array(color) + intensity_var, 0, 255).tolist())

                if ctype == 'abnormal':
                    # Irregular shape for abnormal cells
                    pts = []
                    for angle in np.linspace(0, 2*np.pi, 16):
                        jr = r * (0.7 + 0.4 * np.random.rand())
                        pts.append([int(cx + jr*np.cos(angle)), int(cy + jr*np.sin(angle))])
                    pts = np.array(pts)
                    cv2.fillPoly(img, [pts], cell_color)
                    # Nucleus (darker center)
                    nucleus_color = tuple(np.clip(np.array(color) // 2, 0, 255).tolist())
                    cv2.circle(img, (cx, cy), r//3, nucleus_color, -1)
                else:
                    cv2.circle(img, (cx, cy), r, cell_color, -1)
                    # Nucleus
                    if ctype == 'normal':
                        cv2.circle(img, (cx+np.random.randint(-3,3), cy+np.random.randint(-3,3)),
                                  r//3, (0, 100, 20), -1)

                break

    return img, cells


cell_img, cell_list = create_cell_image(n_cells=25, img_size=512)

# Visualize with ground truth masks
gt_annotated = cell_img.copy()
type_colors = {'normal': (0,255,100), 'abnormal': (0,100,255), 'dead': (100,100,255)}
counts = {'normal': 0, 'abnormal': 0, 'dead': 0}

for cx, cy, r, ctype in cell_list:
    color = type_colors[ctype]
    cv2.circle(gt_annotated, (cx, cy), r+3, color, 2)
    cv2.putText(gt_annotated, ctype[0].upper(), (cx-5, cy+5),
               cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    counts[ctype] += 1

# Legend
for i, (ctype, color) in enumerate(type_colors.items()):
    cv2.rectangle(gt_annotated, (10, 10+i*25), (30, 28+i*25), color, -1)
    cv2.putText(gt_annotated, f'{ctype}: {counts[ctype]}', (35, 24+i*25),
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(cv2.cvtColor(cell_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Synthetic Fluorescence Microscopy Image\n(simulating real cell imaging)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(gt_annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Ground Truth Annotations\n'
                  f'N={counts["normal"]} normal, A={counts["abnormal"]} abnormal, '
                  f'D={counts["dead"]} dead', fontsize=11)
axes[1].axis('off')

plt.suptitle('Cell Instance Segmentation — Mini Project', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Image has {len(cell_list)} cells total: "
      f"{counts['normal']} normal, {counts['abnormal']} abnormal, {counts['dead']} dead")

In [ ]:
# ── Simulate Detectron2 output: per-cell masks and measurements ───────────────

def simulate_detectron2_cell_detection(img, cells, fp_rate=0.05, fn_rate=0.10):
    """
    Simulate what Detectron2 Mask R-CNN would output for cell segmentation.
    In production: replace with actual predictor(img) call.

    Returns:
        detected: list of (cx, cy, r, class_name, confidence, mask)
    """
    detected = []

    for cx, cy, r, ctype in cells:
        if np.random.rand() < fn_rate:
            continue   # simulate missed detection

        conf = 0.75 + np.random.rand() * 0.22

        # Simulate slight box jitter
        dcx = cx + np.random.randint(-3, 3)
        dcy = cy + np.random.randint(-3, 3)
        dr  = r  + np.random.randint(-2, 3)

        # Create binary mask for this cell
        mask = np.zeros(img.shape[:2], dtype=bool)
        ys, xs = np.ogrid[:img.shape[0], :img.shape[1]]
        mask[(xs-dcx)**2 + (ys-dcy)**2 <= dr**2] = True

        # Simulate class prediction (mostly correct with occasional error)
        pred_class = ctype
        if np.random.rand() < 0.08:  # 8% classification error
            pred_class = np.random.choice([t for t in ['normal','abnormal','dead'] if t != ctype])

        detected.append((dcx, dcy, dr, pred_class, conf, mask))

    # Add false positives
    n_fp = int(len(cells) * fp_rate)
    for _ in range(n_fp):
        r_fp = np.random.randint(8, 20)
        cx_fp = np.random.randint(r_fp, img.shape[1]-r_fp)
        cy_fp = np.random.randint(r_fp, img.shape[0]-r_fp)
        mask_fp = np.zeros(img.shape[:2], dtype=bool)
        ys, xs = np.ogrid[:img.shape[0], :img.shape[1]]
        mask_fp[(xs-cx_fp)**2 + (ys-cy_fp)**2 <= r_fp**2] = True
        detected.append((cx_fp, cy_fp, r_fp, 'dead', 0.55, mask_fp))

    return detected


detections = simulate_detectron2_cell_detection(cell_img, cell_list)

# Visualize predictions
pred_vis = cell_img.copy()
mask_overlay = np.zeros_like(cell_img)

type_colors_bgr = {
    'normal':   (50, 255, 50),
    'abnormal': (255, 50, 50),
    'dead':     (50, 50, 200),
}

cell_measurements = []
pred_counts = {'normal': 0, 'abnormal': 0, 'dead': 0}

for cx, cy, r, ctype, conf, mask in detections:
    color = type_colors_bgr[ctype]
    mask_overlay[mask] = color

    # Measure cell properties
    area = np.sum(mask)  # pixel area
    perimeter_approx = 2 * np.pi * r  # approximate
    circularity = (4 * np.pi * area) / (perimeter_approx**2 + 1e-6)

    cell_measurements.append({
        'type': ctype, 'conf': conf,
        'area_px': area, 'radius_px': r,
        'circularity': min(circularity, 1.0),
    })

    pred_counts[ctype] = pred_counts.get(ctype, 0) + 1

pred_vis = cv2.addWeighted(pred_vis, 0.55, mask_overlay, 0.45, 0)

# Draw box outlines
for cx, cy, r, ctype, conf, mask in detections:
    color = type_colors_bgr[ctype]
    cv2.circle(pred_vis, (cx, cy), r+3, color, 2)

# Summary box
for i, (ctype, count) in enumerate(pred_counts.items()):
    color = type_colors_bgr[ctype]
    cv2.rectangle(pred_vis, (350, 10+i*25), (370, 28+i*25), color, -1)
    cv2.putText(pred_vis, f'{ctype}: {count}', (375, 24+i*25),
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)


# Plot predictions + measurements
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(cv2.cvtColor(pred_vis, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Detected {len(detections)} cells\n(green=normal, red=abnormal, blue=dead)', fontsize=10)
axes[0].axis('off')

# Area distribution by type
areas_by_type = {t: [m['area_px'] for m in cell_measurements if m['type']==t] for t in pred_counts}
colors_list = ['green', 'red', 'blue']
for i, (t, areas) in enumerate(areas_by_type.items()):
    if areas:
        axes[1].hist(areas, bins=15, alpha=0.6, color=colors_list[i], label=f'{t} (n={len(areas)})')
axes[1].set_xlabel('Cell area (pixels²)')
axes[1].set_ylabel('Count')
axes[1].set_title('Cell Size Distribution\n(pathologists use this to detect abnormalities)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Circularity vs area
for t, color in zip(pred_counts.keys(), colors_list):
    meas = [m for m in cell_measurements if m['type'] == t]
    if meas:
        axes[2].scatter([m['area_px'] for m in meas],
                       [m['circularity'] for m in meas],
                       c=color, alpha=0.7, label=t, s=60, edgecolors='white', lw=0.5)
axes[2].set_xlabel('Area (px²)')
axes[2].set_ylabel('Circularity (1=perfect circle)')
axes[2].set_title('Area vs Circularity\nAbnormal cells = larger, less circular')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Cell Instance Segmentation — Detectron2 Mask R-CNN Output Analysis', fontsize=12)
plt.tight_layout()
plt.show()

# Summary report
print("\n" + "═"*55)
print("  PATHOLOGY REPORT")
print("═"*55)
print(f"  Total cells detected: {len(detections)}")
for ctype, count in pred_counts.items():
    pct = count / max(1, len(detections)) * 100
    areas = [m['area_px'] for m in cell_measurements if m['type'] == ctype]
    avg_area = np.mean(areas) if areas else 0
    print(f"  {ctype:10s}: {count:3d} cells ({pct:5.1f}%) | avg area: {avg_area:.0f}px²")
print()
abnormal_pct = pred_counts.get('abnormal', 0) / max(1, len(detections)) * 100
print(f"  DIAGNOSIS: {'⚠ ABNORMAL CELLS DETECTED — refer to specialist' if abnormal_pct > 15 else 'Normal range — routine follow-up'}")

---
## 10. Detectron2 vs YOLO — When to Use Which <a id='10-comparison'></a>

In [ ]:
comparison = [
    ('Speed (inference)',  'Very fast (30-100 FPS CPU)', 'Slower (5-15 FPS GPU needed)'),
    ('Accuracy',          'Very good (mAP ~45-50)',    'Best (mAP ~47-55)'),
    ('Ease of use',       'Extremely simple API',      'Complex, more configuration'),
    ('Instance masks',    'YOLOv8-seg: yes',           'Yes, pixel-perfect'),
    ('Pose estimation',   'YOLOv8-pose: yes',          'Yes, 17 keypoints'),
    ('Small objects',     'OK',                        'Better (FPN + RoI Align)'),
    ('Custom training',   'Very easy (YAML + folder)', 'More setup (registry)'),
    ('Mobile deployment', 'Excellent (TFLite/ONNX)',   'Harder (PyTorch only)'),
    ('Community/support', 'Large, active',             'Research-focused'),
    ('Installation',      'pip install ultralytics',   'Build from source'),
    ('GPU required?',     'No (CPU mode works)',       'Strongly recommended'),
]

print(f"{'Aspect':25s} {'YOLO (Ultralytics)':30s} {'Detectron2'}")
print("─" * 90)
for aspect, yolo, d2 in comparison:
    print(f"{aspect:25s} {yolo:30s} {d2}")

print("\n" + "═"*90)
print("\nRECOMMENDATION GUIDE:")
print()
print("  Use YOLO when:")
print("    • Real-time inference (video, webcam, edge device)")
print("    • Quick prototyping / competition")
print("    • Mobile/embedded deployment")
print("    • Small team, need fast iteration")
print("    • Dataset < 50K images")
print()
print("  Use Detectron2 when:")
print("    • Research publication (standard benchmark)")
print("    • Need pixel-precise segmentation quality")
print("    • Dense, overlapping, or very small objects")
print("    • Complex multi-task setup (detection + keypoints + masks)")
print("    • Server-side batch processing (speed not critical)")
print()
print("  In 2024+, for MOST production use cases: YOLO is the better default.")
print("  Detectron2 shines in research and high-precision medical/scientific imaging.")

---
## 11. Common Pitfalls <a id='11-pitfalls'></a>

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║             DETECTRON2 — COMMON PITFALLS                         ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. WRONG: Feeding RGB images to predictor                       ║
║     im = plt.imread('image.jpg')    ← RGB (H,W,3) float          ║
║     outputs = predictor(im)         ← WRONG! expects BGR uint8   ║
║  RIGHT: Always use cv2.imread() for Detectron2                  ║
║     im = cv2.imread('image.jpg')    ← BGR uint8 ✓                ║
║     outputs = predictor(im)                                      ║
║                                                                  ║
║  2. WRONG: Forgetting to set NUM_CLASSES for custom training     ║
║     # Default: 80 COCO classes                                   ║
║     # Model head outputs 80 classes → wrong for custom data!     ║
║  RIGHT:                                                          ║
║     cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(your_classes)  ✓      ║
║                                                                  ║
║  3. WRONG: Using COCO weights without transfer learning setup    ║
║     cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(...)        ║
║     # The final detection head will be randomly re-initialized   ║
║     # because NUM_CLASSES changed — that's actually CORRECT      ║
║     # BUT: use a smaller LR to not destroy the backbone          ║
║  RIGHT: cfg.SOLVER.BASE_LR = 0.00025  (default is fine)         ║
║                                                                  ║
║  4. WRONG: Running out of GPU memory                            ║
║     cfg.SOLVER.IMS_PER_BATCH = 16   ← OOM on 8GB GPU            ║
║  RIGHT: Start with 2 images per batch                           ║
║     cfg.SOLVER.IMS_PER_BATCH = 2                                 ║
║     cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 64  (reduce too) ║
║                                                                  ║
║  5. WRONG: Not matching annotation format to BoxMode            ║
║     bbox = [x1, y1, x2, y2]  but mode = BoxMode.XYWH_ABS       ║
║     → silently wrong boxes!                                      ║
║  RIGHT: Match your format exactly:                               ║
║     'bbox_mode': BoxMode.XYXY_ABS   for [x1,y1,x2,y2]          ║
║     'bbox_mode': BoxMode.XYWH_ABS   for [x,y,width,height]      ║
║                                                                  ║
║  6. WRONG: Forgetting to move instances to CPU before numpy     ║
║     masks = outputs['instances'].pred_masks.numpy()  ← FAILS GPU ║
║  RIGHT:                                                          ║
║     inst = outputs['instances'].to('cpu')                        ║
║     masks = inst.pred_masks.numpy()  ✓                           ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
## 12. Interview Q&A <a id='12-interview'></a>

In [ ]:
qa = [
    ("What is the difference between semantic and instance segmentation?",
     "Semantic segmentation: assigns a class label to every pixel. All pixels of the same "
     "class get the same label — you can't distinguish individual objects (two touching cars "
     "merge into one region). "
     "Instance segmentation: detects each object individually and assigns each a unique mask. "
     "Two touching cars get two separate masks. "
     "Panoptic segmentation: combines both — instance segmentation for 'things' (countable "
     "objects like people, cars) + semantic segmentation for 'stuff' (uncountable background "
     "like sky, road, grass)."),

    ("How does Mask R-CNN extend Faster R-CNN?",
     "Faster R-CNN: (1) ResNet+FPN backbone; (2) RPN proposes regions; "
     "(3) RoI Pooling (or Align) extracts features; (4) Box Head outputs class + box. "
     "Mask R-CNN adds a parallel Mask Head: a small FCN (Fully Convolutional Network) "
     "that takes the RoI-Aligned feature and outputs a 28×28 binary mask for each class. "
     "Key change: RoI Align (vs Pooling) uses bilinear interpolation to avoid quantization "
     "errors — critical for pixel-precise masks. The mask head runs in parallel with the "
     "box head, adding minimal overhead (~20% slower than Faster R-CNN)."),

    ("What is RoI Align and why is it better than RoI Pooling?",
     "Both extract a fixed-size feature map from a proposed region. "
     "RoI Pooling: quantizes (rounds) the region boundaries to integer pixel coordinates. "
     "This misalignment is fine for classification but terrible for precise masks. "
     "RoI Align: uses bilinear interpolation to sample features at exact non-integer "
     "positions — no rounding, no misalignment. Critical for Mask R-CNN because the "
     "28×28 mask must perfectly align with the actual object pixels. "
     "Improves mask AP by ~3-4 points on COCO benchmark."),

    ("What is FPN (Feature Pyramid Network) and why is it important?",
     "Traditional CNNs produce features at different scales internally, but only use the "
     "last (most downsampled) feature map for detection. This misses small objects. "
     "FPN creates a top-down pathway that merges high-level semantics (deep layers, coarse "
     "resolution) with low-level detail (shallow layers, fine resolution). "
     "Each scale of the pyramid detects objects at its appropriate size: "
     "P2 (finest) for tiny objects, P5 (coarsest) for large objects. "
     "Result: uniform detection performance across all object scales."),

    ("Explain the Region Proposal Network (RPN).",
     "RPN is a small fully-convolutional network that slides over the feature map and, "
     "at each location, predicts: (1) objectness score (is there an object here?); "
     "(2) 4 box coordinates for k anchor boxes at each position. "
     "Anchors: predefined boxes of different scales and aspect ratios centered at each location. "
     "Top-N proposals (e.g., 300) by objectness score are passed to the detection head. "
     "Replaces the old sliding window approach (which needed to run a classifier at every "
     "position) with a lightweight single forward pass."),

    ("When would you choose Detectron2 over YOLO in production?",
     "Choose Detectron2 when: (1) Pixel-precise instance segmentation is required "
     "(medical imaging, AR effects, precise area measurement); (2) Small or dense overlapping "
     "objects (FPN + RoI Align handles these better); (3) Research requiring standard COCO "
     "benchmarks; (4) Multi-task outputs (detection + keypoints + segmentation simultaneously). "
     "Choose YOLO when: (1) Real-time requirements (>30 FPS); (2) Edge/mobile deployment; "
     "(3) Simple API and fast iteration needed; (4) Most production applications where "
     "speed matters more than marginal accuracy gains."),
]

print("=" * 70)
print("  INTERVIEW Q&A — DETECTRON2 / INSTANCE SEGMENTATION")
print("=" * 70)
for i, (q, a) in enumerate(qa, 1):
    print(f"\nQ{i}: {q}")
    print(f"\nA{i}: {a}")
    print("\n" + "─" * 70)

---
## 13. Resources <a id='13-resources'></a>

### Official Documentation
- **Detectron2 docs**: https://detectron2.readthedocs.io/
- **Model Zoo**: https://github.com/facebookresearch/detectron2/blob/main/MODEL_ZOO.md
- **Tutorials**: https://detectron2.readthedocs.io/en/latest/tutorials/

### Research Papers
- **Mask R-CNN** (He et al., 2017): https://arxiv.org/abs/1703.06870
- **FPN** (Lin et al., 2017): https://arxiv.org/abs/1612.03144
- **Faster R-CNN** (Ren et al., 2015): https://arxiv.org/abs/1506.01497
- **Panoptic FPN**: https://arxiv.org/abs/1901.02446

### Video Tutorials
- **Detectron2 custom training** (YouTube): https://youtu.be/GoItxr16ae8
- **Instance segmentation explained**: https://youtu.be/lMqxFNGJYAw
- **FAIR talks on Detectron2**: https://youtu.be/0e4IoN5H7is

### Related Libraries
- **MMDetection** (OpenMMLab): https://github.com/open-mmlab/mmdetection
- **Supervision** (post-processing): https://supervision.roboflow.com/
- **SAM** (Segment Anything Model by Meta): https://github.com/facebookresearch/segment-anything

## Interview Questions & Answers

---

**Q1: What is the difference between Detectron2's Mask R-CNN and YOLO architectures?**

A: **Mask R-CNN** (Detectron2) is a **two-stage** detector: Stage 1 proposes regions likely to contain objects (Region Proposal Network), Stage 2 classifies and refines them + produces pixel masks. More accurate but slower (~5 FPS). **YOLO** is **one-stage**: directly predicts bounding boxes and classes on a grid — much faster (30-100 FPS) but historically less accurate on small objects. Detectron2 is preferred for research/high-accuracy tasks; YOLO for real-time applications.

---

**Q2: What is instance segmentation vs semantic segmentation?**

A: **Semantic segmentation** labels every pixel with a class (all cars = one colour, all trees = another). It cannot distinguish between two cars touching each other. **Instance segmentation** assigns a unique mask to each individual object (car 1 = red, car 2 = blue). Detectron2's Mask R-CNN does instance segmentation — it knows there are 3 separate people, not just "there are people here." Panoptic segmentation combines both: "stuff" (sky, road) is semantic, "things" (people, cars) is instance.

---

**Q3: What is a Feature Pyramid Network (FPN) and why does Detectron2 use it?**

A: A FPN builds a multi-scale feature hierarchy from a single backbone. The backbone (ResNet) produces feature maps at different resolutions (coarse→fine). FPN adds top-down pathways to inject high-level semantic information into fine-grained maps. Result: the detector can detect both tiny objects (using fine, high-res maps) and large objects (using coarse, semantic maps) simultaneously. Without FPN, detecting a pedestrian far away and a truck up close in the same image would require much larger models.

---

**Q4: How does COCO evaluation work? What does AP@50:95 mean?**

A: COCO's primary metric is **AP@[.50:.95]** — Average Precision averaged over 10 IoU thresholds from 0.50 to 0.95 in steps of 0.05. IoU (Intersection over Union) measures how much the predicted box overlaps the ground-truth box. AP@50 (PASCAL VOC style) only requires 50% overlap. COCO's stricter metric rewards precise localisation. A model scoring 50 AP@50 but 30 AP@50:95 is finding objects but not precisely boxing them — important for robotics/surgery where exact boundaries matter.

---

**Q5: When would you choose to fine-tune a Detectron2 model vs train from scratch?**

A: Almost always fine-tune. Detectron2's pretrained models are trained on COCO (330K images, 80 classes). Fine-tuning gives you: (1) faster convergence (hours vs weeks); (2) better accuracy with less data (transfer learning captures universal visual features like edges, textures, shapes); (3) lower GPU cost. Train from scratch only when your domain is radically different (satellite imagery, medical scans, industrial X-rays) AND you have 100K+ labelled examples. Even then, initialise from ImageNet weights.

---

**Q6: What is the role of Non-Maximum Suppression (NMS) in object detection?**

A: Detectors generate many overlapping bounding box proposals. NMS removes duplicates: (1) rank boxes by confidence score; (2) keep the highest-scoring box; (3) remove any box with IoU > threshold (e.g., 0.5) with the kept box; (4) repeat. Without NMS, one car might generate 50 overlapping boxes. Soft-NMS is a variant that decays scores of nearby boxes instead of removing them — better for crowds where true objects overlap (e.g., a concert). Detectron2 applies NMS after both the RPN and the final classifier stages.

## Recommended Resources

| Resource | Link |
|---|---|
| Detectron2 GitHub | https://github.com/facebookresearch/detectron2 |
| Mask R-CNN Paper | https://arxiv.org/abs/1703.06870 |
| FPN Paper | https://arxiv.org/abs/1612.03144 |
| COCO Dataset | https://cocodataset.org/ |
| Model Zoo | https://github.com/facebookresearch/detectron2/blob/main/MODEL_ZOO.md |


---
## 14. Summary & What's Next <a id='14-summary'></a>

### What You Learned

| Concept | Key Takeaway |
|---|---|
| Segmentation types | Semantic (class-level) vs Instance (object-level) vs Panoptic (both) |
| Mask R-CNN | Faster R-CNN + FPN + RoI Align + parallel Mask Head |
| FPN | Multi-scale feature pyramid — detects small AND large objects |
| RoI Align | Bilinear interpolation — no quantization errors → precise masks |
| RPN | Lightweight objectness + box predictor → 300 candidate regions |
| DefaultPredictor | 3-line inference: `get_cfg()` → `merge_from_file()` → `DefaultPredictor(cfg)` |
| Config system | Hierarchical YAML — merge + override specific values |
| Custom training | Register dataset → set NUM_CLASSES → `DefaultTrainer.train()` |
| YOLO vs D2 | Speed vs precision — use YOLO for production, D2 for research/medical |

### Congratulations — Computer Vision Phase Complete!

You now know the complete CV stack:
1. **OpenCV** — image manipulation (pixels, filters, edges, contours)
2. **Torchvision** — deep CNN training, transfer learning, Grad-CAM
3. **YOLO** — real-time object detection + tracking
4. **Detectron2** — research-grade segmentation and keypoint detection

### What's Next — Reinforcement Learning & Generative AI

| Phase | Topics |
|---|---|
| Phase 5 | Reinforcement Learning (Q-Learning, PPO, OpenAI Gym) |
| Phase 5 | Generative AI (GANs, VAEs, Diffusion Models, LLMs) |
| Phase 6 | MLOps (experiment tracking, model deployment, monitoring) |

**You now have a complete foundation in Computer Vision — from pixels to pixel-perfect segmentation!**